# Greeks playground: three sliders, one book

An option's price depends on three things that move: the **future price**, the **vol** and
**time**. Each slider below moves one of them, the others stay where they are. The charts and the
numbers show how the book reacts:

- **delta, gamma**: the future price moves;
- **vega**: the vol moves;
- **theta**: time passes.

The book is **delta-hedged with futures at the start** (future 75 $, vol 32 %, today). From
there, the hedge is not touched: what the sliders show is what the hedge leaves.

1. One option, delta-hedged
2. A book of five options, delta-hedged

The sliders need a running notebook (`pip install ipywidgets`); on GitHub only the starting
picture shows.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql
from ipywidgets import FloatSlider, IntSlider, interact

from cmdty import MarketData, Quote
from cmdty.instruments import VanillaOption
from cmdty.models.black76 import analytic_greeks, expected_payoff
from cmdty.plotting import (MUTED, SERIES, plot_bars, plot_delta, plot_gamma, plot_theta,
                            plot_value, plot_vega, use_style)

use_style()


def market_at(f: float, vol: float, date: ql.Date) -> MarketData:
    """The market for one state of the three sliders."""
    return MarketData(today=date, rate=0.04, quotes={U: Quote(f, vol)})


def opt(kind: str, strike: float, qty: float, months: int) -> VanillaOption:
    """One option on the Brent future, qty in bbl (negative = sold), expiring in `months`."""
    return VanillaOption(underlying=U, contract=CONTRACT, strike=strike, option_type=kind,
                         quantity=qty, expiry=TODAY + ql.Period(months, ql.Months))


def book_value(book, f: float, vol: float, date: ql.Date) -> float:
    """Value of all the options, $ (Black-76, QuantLib, undiscounted)."""
    m = market_at(f, vol, date)
    return sum(o.quantity * expected_payoff(o, m) for o in book)


def book_greeks(book, f: float, vol: float, date: ql.Date, futures: float = 0.0) -> dict:
    """Greeks of the options plus the futures hedge (futures only add delta)."""
    m = market_at(f, vol, date)
    g = [(o.quantity, analytic_greeks(o, m)) for o in book]
    theta = book_value(book, f, vol, date + 1) - book_value(book, f, vol, date)
    return dict(delta=sum(q * x.delta for q, x in g) + futures,
                gamma=sum(q * x.gamma for q, x in g),
                vega=sum(q * x.vega for q, x in g), theta=theta)

In [2]:
TODAY = ql.Date(25, 11, 2026)
F0, VOL = 75.0, 0.32  # where the hedge is set: future price, implied vol
U, CONTRACT = "BRENT", "2027-03"
GRID = np.linspace(55, 95, 81)  # future prices on the x axis

The whole playground is one function: give it a book, it hedges the delta with futures at the
start and draws six panels for the slider values.

- **Hedged P&L**: the hedged book against the future price, now (blue) and at the start (dashed grey).
- **Delta, gamma, vega, theta** of the hedged book, against the future price.
- **P&L explain**: the P&L since the start, split by greek (greeks taken at the start), next to
  the true P&L from a full repricing. The gap is the residual.

A white dot marks the slider's future price on every curve.

In [3]:
def playground(book, max_days: int):
    """Hedge `book` with futures at the start, then sliders for future price, vol and time."""
    start = book_greeks(book, F0, VOL, TODAY)
    futures = -start["delta"]  # the delta hedge, never touched afterwards
    v_start = book_value(book, F0, VOL, TODAY)
    pnl_start = [book_value(book, f, VOL, TODAY) - v_start + futures * (f - F0) for f in GRID]
    print(f"delta hedge at the start: {futures:+,.0f} bbl of futures")

    def show(future=F0, vol=VOL, days=0):
        date = TODAY + days
        pnl = [book_value(book, f, vol, date) - v_start + futures * (f - F0) for f in GRID]
        g = [book_greeks(book, f, vol, date, futures) for f in GRID]
        curves = {k: np.array([x[k] for x in g]) for k in ("delta", "gamma", "vega", "theta")}
        now = book_greeks(book, future, vol, date, futures)
        true_pnl = book_value(book, future, vol, date) - v_start + futures * (future - F0)

        # P&L explain from the start: each greek times the move of its own input
        dF, dvol = future - F0, (vol - VOL) * 100
        explain = {"delta": (start["delta"] + futures) * dF,  # 0: hedged
                   "gamma": 0.5 * start["gamma"] * dF ** 2,
                   "vega": start["vega"] * dvol,
                   "theta": start["theta"] * days}
        explain["residual"] = true_pnl - sum(explain.values())
        explain["true P&L"] = true_pnl

        fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
        kw = dict(f0=F0, xlabel="future price, $/bbl")
        plot_value(GRID, {"start": pnl_start, "now": pnl}, title="Hedged P&L", ylabel="P&L, $",
                   subtitle=f"{true_pnl:+,.0f} at future {future:.1f}", ax=axes[0, 0],
                   styles={"start": {"color": MUTED, "linestyle": (0, (4, 3))},
                           "now": {"color": SERIES[0]}}, **kw)
        axes[0, 0].plot([future], [true_pnl], "o", color="white", zorder=3)
        panels = [("delta", plot_delta), ("gamma", plot_gamma), ("vega", plot_vega),
                  ("theta", plot_theta)]
        for ax, (name, plot) in zip(axes.flat[1:], panels):
            plot(GRID, curves[name], subtitle=f"{now[name]:+,.1f} at future {future:.1f}", ax=ax, **kw)
            ax.plot([future], [now[name]], "o", color="white", zorder=3)
        plot_bars(explain, ax=axes[1, 2], title="P&L explain since the start", ylabel="$",
                  subtitle=f"move {dF:+.1f} $, vol {dvol:+.0f} pts, {days} days",
                  colors=[SERIES[1], SERIES[2], SERIES[3], SERIES[0], MUTED, "white"])
        axes[1, 2].tick_params(axis="x", labelrotation=30)
        fig.tight_layout(h_pad=2)
        plt.show()

    interact(show,
             future=FloatSlider(value=F0, min=60, max=90, step=0.5, description="future $",
                                continuous_update=False),
             vol=FloatSlider(value=VOL, min=0.15, max=0.60, step=0.01, description="vol",
                             readout_format=".0%", continuous_update=False),
             days=IntSlider(value=0, min=0, max=max_days, description="days passed",
                            continuous_update=False))

## 1. One option, delta-hedged

The long call 83 from notebook 07, 10,000 bbl, 3 months.

Try one slider at a time:

- **future only**: the P&L smiles (long gamma), the delta drifts away from 0;
- **days only**: the whole P&L curve sinks, theta;
- **vol only**: the whole curve rises or falls, vega;
- then all three: the explain bars still add up, with a small residual.

In [4]:
book1 = [opt("call", 83.0, +10_000, 3)]
playground(book1, max_days=80)

delta hedge at the start: -2,910 bbl of futures


interactive(children=(FloatSlider(value=75.0, continuous_update=False, description='future $', max=90.0, min=6…

## 2. A book of five options, delta-hedged

Closer to a desk: several strikes, bought and sold, two maturities.

| Trade | Why it is there |
|---|---|
| long 10,000 call 83, 3m | the call of part 1 |
| short 15,000 call 90, 3m | sold upside, collects premium |
| long 8,000 put 70, 3m | downside protection |
| short 5,000 put 65, 6m | sold far put, longer maturity: vega |
| long 6,000 call 75, 1m | short-dated at the money: gamma |

The greeks of a book are the **sums** of the greeks of its options. Bought and sold options
offset each other, so the book can be long gamma at one price and short gamma at another.

In [5]:
book2 = [
    opt("call", 83.0, +10_000, 3),
    opt("call", 90.0, -15_000, 3),
    opt("put", 70.0, +8_000, 3),
    opt("put", 65.0, -5_000, 6),
    opt("call", 75.0, +6_000, 1),
]


def label(o: VanillaOption) -> str:
    side = "long" if o.quantity > 0 else "short"
    return f"{side} {abs(o.quantity):,.0f} {o.option_type.value} {o.strike:.0f}"


rows = [dict(trade=label(o), expiry=o.expiry.ISO(), **book_greeks([o], F0, VOL, TODAY))
        for o in book2]
rows.append(dict(trade="book", expiry="", **book_greeks(book2, F0, VOL, TODAY)))
pd.DataFrame(rows).set_index("trade").round(1)

,expiry,delta,gamma,vega,theta
trade,,,,,
"long 10,000 call 83",2027-02-25,2909.8,284.5,1290.9,-224.9
"short 15,000 call 90",2027-02-25,-2187.3,-284.8,-1292.2,224.6
"long 8,000 put 70",2027-02-25,-2440.8,232.6,1055.3,-183.9
"short 5,000 put 65",2027-05-25,1136.6,-89.2,-796.6,70.5
"long 6,000 call 75",2026-12-25,3109.8,347.5,514.1,-276.5
book,,2527.9,490.6,771.6,-390.3


In [6]:
playground(book2, max_days=29)  # the 1-month call expires after that

delta hedge at the start: -2,528 bbl of futures


interactive(children=(FloatSlider(value=75.0, continuous_update=False, description='future $', max=90.0, min=6…

Things to look for:

- The **gamma** of the book changes sign along the price axis: the bought options dominate in
  one zone, the sold ones in another.
- **Theta** follows gamma with the opposite sign: where the book is long gamma it pays theta,
  where it is short gamma it collects.
- **Vega** is not proportional to gamma any more: the 6-month put carries more vega per unit of
  gamma than the 1-month call. Two maturities, two different greek profiles.

Next: the same P&L explain day by day over several weeks, with the delta re-hedged every day.